<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 06 - Procesamiento en Tiempo Real

**Módulo:** Sistemas de Big Data (SBD - 5074)

**Contenido:** Batch vs Streaming, conceptos de tiempo real, simulación

---

## Objetivos
- Diferenciar entre batch processing y stream processing
- Comprender conceptos de ventanas temporales
- Simular procesamiento en tiempo real con Python

## Batch vs Streaming

| Aspecto | Batch | Streaming |
|---------|-------|----------|
| Datos | Estáticos, finitos | En movimiento, infinitos |
| Latencia | Alta (min-horas) | Baja (ms-segundos) |
| Procesamiento | Por lotes | Evento a evento |
| Casos de uso | ETL, informes | Fraude, alertas |

## 1. Simulación de Stream de Datos

In [ ]:
import time
import random
from datetime import datetime
from collections import deque

class StreamSimulator:
    """Simula un stream de eventos de transacciones"""
    
    def __init__(self):
        self.usuarios = [f"user_{i}" for i in range(1, 101)]
        self.productos = ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares']
    
    def generar_evento(self):
        """Genera un evento aleatorio"""
        return {
            'timestamp': datetime.now().isoformat(),
            'event_time': time.time(),
            'user_id': random.choice(self.usuarios),
            'producto': random.choice(self.productos),
            'cantidad': random.randint(1, 5),
            'precio': round(random.uniform(10, 500), 2)
        }
    
    def stream(self, n_eventos=10, delay_max=1.0):
        """Genera n eventos con delay aleatorio"""
        for i in range(n_eventos):
            evento = self.generar_evento()
            yield evento
            time.sleep(random.uniform(0.1, delay_max))

# Demostración
print("=== SIMULACIÓN DE STREAM ===")
simulator = StreamSimulator()

for i, evento in enumerate(simulator.stream(n_eventos=5, delay_max=0.5)):
    print(f"\nEvento {i+1}:")
    print(f"  Timestamp: {evento['timestamp']}")
    print(f"  Usuario: {evento['user_id']}")
    print(f"  Producto: {evento['producto']} x{evento['cantidad']}")
    print(f"  Precio: €{evento['precio']}")

## 2. Ventanas Temporales (Tumbling Window)

In [ ]:
class TumblingWindow:
    """Implementa ventanas temporales fijas (tumbling)"""
    
    def __init__(self, window_size_seconds):
        self.window_size = window_size_seconds
        self.current_window = []
        self.window_start = time.time()
    
    def add(self, evento):
        """Añade evento a la ventana actual"""
        current_time = time.time()
        
        # ¿Se cerró la ventana?
        if current_time - self.window_start >= self.window_size:
            resultado = self.cerrar_ventana()
            self.window_start = current_time
            self.current_window = [evento]
            return resultado
        else:
            self.current_window.append(evento)
            return None
    
    def cerrar_ventana(self):
        """Procesa y cierra la ventana actual"""
        if not self.current_window:
            return None
        
        total_ventas = sum(e['precio'] * e['cantidad'] for e in self.current_window)
        num_eventos = len(self.current_window)
        
        return {
            'ventana_inicio': datetime.fromtimestamp(self.window_start).isoformat(),
            'num_eventos': num_eventos,
            'total_ventas': round(total_ventas, 2)
        }

# Demostración con ventanas de 2 segundos
print("=== VENTANAS TEMPORALES (2 segundos) ===")
simulator = StreamSimulator()
window = TumblingWindow(window_size_seconds=2)

for evento in simulator.stream(n_eventos=15, delay_max=0.5):
    resultado = window.add(evento)
    if resultado:
        print(f"\n🔔 VENTANA CERRADA:")
        print(f"   Inicio: {resultado['ventana_inicio']}")
        print(f"   Eventos: {resultado['num_eventos']}")
        print(f"   Total ventas: €{resultado['total_ventas']}")

# Cerrar última ventana
resultado_final = window.cerrar_ventana()
if resultado_final:
    print(f"\n🔔 VENTANA FINAL:")
    print(f"   Eventos: {resultado_final['num_eventos']}")
    print(f"   Total: €{resultado_final['total_ventas']}")

## 3. Detección de anomalías en tiempo real

In [ ]:
class AnomalyDetector:
    """Detecta transacciones anómalas en tiempo real"""
    
    def __init__(self, threshold_amount=400, max_per_user_per_minute=3):
        self.threshold = threshold_amount
        self.max_per_user = max_per_user_per_minute
        self.user_history = {}  # user_id -> lista de timestamps
    
    def check(self, evento):
        """Verifica si el evento es anómalo"""
        alertas = []
        user_id = evento['user_id']
        total = evento['precio'] * evento['cantidad']
        current_time = evento['event_time']
        
        # Regla 1: Monto alto
        if total > self.threshold:
            alertas.append(f"⚠️ MONTO ALTO: {user_id} - €{total:.2f}")
        
        # Regla 2: Frecuencia alta por usuario
        if user_id not in self.user_history:
            self.user_history[user_id] = deque(maxlen=100)
        
        # Limpiar eventos antiguos (> 60 segundos)
        while self.user_history[user_id] and current_time - self.user_history[user_id][0] > 60:
            self.user_history[user_id].popleft()
        
        self.user_history[user_id].append(current_time)
        
        if len(self.user_history[user_id]) > self.max_per_user:
            alertas.append(f"⚠️ FRECUENCIA ALTA: {user_id} - {len(self.user_history[user_id])} transacciones/min")
        
        return alertas

# Demostración
print("=== DETECCIÓN DE ANOMALÍAS EN TIEMPO REAL ===")
simulator = StreamSimulator()
detector = AnomalyDetector(threshold_amount=300, max_per_user_per_minute=2)

transacciones_normales = 0
alertas_generadas = 0

for evento in simulator.stream(n_eventos=20, delay_max=0.3):
    alertas = detector.check(evento)
    
    if alertas:
        alertas_generadas += len(alertas)
        for alerta in alertas:
            print(alerta)
    else:
        transacciones_normales += 1

print(f"\n=== RESUMEN ===")
print(f"Transacciones normales: {transacciones_normales}")
print(f"Alertas generadas: {alertas_generadas}")

---

## Recursos adicionales

- [Apache Kafka](https://kafka.apache.org/)
- [Spark Streaming](https://spark.apache.org/streaming/)
- [Apache Flink](https://flink.apache.org/)